# 14.02 · Hugging Face: choose, check and run open models

**Module 14 · Local Models & Fine-Tuning** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lesson 14.01 (Ollama, `fits_in_memory`) and Module 06 (embeddings). |
| 🔑 Keys | none for the Hub (public API) · optional `HF_TOKEN` for gated models and paid Inference Providers |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Search** the Hugging Face Hub from Python and read what matters on a model page: licence, gating, task, size.
2. **Shortlist** models for a real use case with explicit rules (commercial licence, no gate, fits your RAM).
3. **Pick** a GGUF quantization by size and run any Hub GGUF in Ollama with `hf.co/<repo>:<quant>`.
4. **Explain** gated models and what an `HF_TOKEN` does (and doesn't) unlock.
5. **Run** an ONNX embedding model and, optionally, a `transformers` chat model on the CPU through LangChain.
6. **Decide** between Ollama, in-process `transformers` and hosted Inference Providers for a given job.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Hub Python library: search](https://huggingface.co/docs/huggingface_hub/guides/search) | `list_models` filters, sorting, `expand` |
| [Model cards](https://huggingface.co/docs/hub/model-cards) and [Licenses](https://huggingface.co/docs/hub/repositories-licenses) | Where the licence lives and what the ids mean |
| [Gated models](https://huggingface.co/docs/hub/models-gated) | `auto` vs `manual` approval, tokens |
| [Use Ollama with any GGUF on the Hub](https://huggingface.co/docs/hub/ollama) | `ollama run hf.co/{user}/{repo}:{quant}` |
| [LangChain Hugging Face integration](https://docs.langchain.com/oss/python/integrations/providers/huggingface) | `ChatHuggingFace`, `HuggingFacePipeline`, `HuggingFaceEndpoint` |
| [Inference Providers pricing](https://huggingface.co/docs/inference-providers/pricing) | What the free tier includes (no monthly credits for free accounts) |

> **Why this matters at work.** Orbita Cloud wants a local model for ticket routing. Before anyone downloads 5 GB, legal asks *"may we use it commercially?"*, IT asks *"does it fit our laptops?"*, and security asks *"who approved access?"*. The Hub answers all three in its metadata; this lesson turns that into a repeatable shortlist instead of a browser tab hunt.

> 🖥️ **CPU-only is fine here.** The Hub calls are small HTTP requests. The local parts use a 67 MB ONNX embedding model and, optionally, a 360M-parameter chat model: seconds on any laptop CPU. Nothing in this lesson needs a GPU or a token.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "huggingface-hub>=1.33.0,<2" "fastembed>=0.9.0,<1" "langchain-ollama>=1.1.0,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "huggingface-hub", "fastembed",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/14-local-models-and-fine-tuning/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'HF_TOKEN')
LANGSMITH_PROJECT = 'm14-hugging-face'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config, look for a local Ollama.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.local import ollama_status

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_HF_TOKEN = is_set("HF_TOKEN")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m14-<lesson>: one tracing project per lesson
OLLAMA = ollama_status(timeout=3.0)                              # never raises; 3 s if nothing is listening
HAVE_OLLAMA = OLLAMA.running

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("local     :", settings.model_local, "| embeddings:", settings.model_local_embed)
print("ollama    :", f"{OLLAMA.version} at {OLLAMA.base_url}, models: {OLLAMA.names() or 'none pulled'}" if HAVE_OLLAMA
      else f"not running at {OLLAMA.base_url}")
if not HAVE_OLLAMA:
    print("\n⚠️ No Ollama server: 🖥️ local cells will print ⏭️ Skipped. Start the Ollama app (or `ollama serve`) and re-run.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from mastery.local import fits_in_memory, license_verdict, load_snapshot, model_details, ollama_hf_tag, quant_files, search_models

SNAPSHOT = load_snapshot(REPO_ROOT / "data" / "samples" / "local" / "hub-snapshot.json")   # recorded 2026-10-16
LIVE = True       # ← False: use only the recorded snapshot (no network)
RAM_GB, VRAM_GB = 32, 0   # ← your machine, as in Lesson 14.01
warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm without ipywidgets: progress bars still print


def hub(repo_id):
    """Live model page when the Hub answers, else the recorded copy (so the lesson runs offline)."""
    if LIVE:
        try:
            return model_details(repo_id)
        except Exception as exc:          # offline, firewall, rate limit: fall back, say so
            print(f"⚠️ Hub unavailable for {repo_id} ({type(exc).__name__}): using the 2026-10-16 snapshot.")
    return SNAPSHOT[repo_id]

---
## Part 1 · Search the Hub like a catalogue

In [ ]:
# 🌐 (LIVE) Most-downloaded Qwen3 text-generation models: public API, no token.
try:
    found = search_models("qwen3", task="text-generation", limit=8) if LIVE else None
except Exception as exc:
    print(f"⚠️ Hub search failed ({type(exc).__name__}): showing the recorded search.")
    found = None
if found is None:
    found = [m for m in SNAPSHOT.values() if m.id.lower().startswith(("qwen/qwen3", "nvidia/qwen3"))][:8]
table([{"model": m.id, "downloads": f"{m.downloads:,}" if m.downloads else "", "license": m.license, "verdict": m.verdict,
        "gated": m.gated} for m in found], ["model", "downloads", "license", "verdict", "gated"])

**You should now see** about eight rows led by `Qwen/Qwen3-0.6B` and `Qwen/Qwen3-4B` with millions of downloads, all `apache-2.0` → `permissive`, `gated False`. Download counts change daily; the snapshot keeps the 2026-10-16 numbers.

### What just happened?

`search_models` wraps `HfApi().list_models(search=..., filter=..., sort="downloads", expand=[...])`. `expand` asks for the card data (licence) and gating in the same request instead of one call per model. Downloads are a popularity signal, not a quality score: the base and instruct variants, quantized copies and fine-tunes of one model all appear separately.

In [ ]:
# 🌐 (LIVE) Five model pages, five kinds of answer: licence, gate and the size of the weights.
REPOS = ["Qwen/Qwen3-4B", "meta-llama/Llama-3.2-1B-Instruct", "google/gemma-3-1b-it",
         "CohereLabs/c4ai-command-r7b-12-2024", "mistralai/Mistral-7B-Instruct-v0.3"]
pages = {r: hub(r) for r in REPOS}
table([{"model": r, "license": m.license, "verdict": m.verdict, "gated": m.gated, "safetensors GB": m.size_gb()}
       for r, m in pages.items()], ["model", "license", "verdict", "gated", "safetensors GB"])

**You should now see** Qwen3-4B `permissive` (≈ 7.5 GB at 16-bit), Llama 3.2 and Gemma 3 `read the terms` with `gated manual`, Command R7B `cc-by-nc-4.0` → `non-commercial`, and Mistral 7B at **27 GB**.

### What just happened?

- **Licence.** `license_verdict` is a first filter, not legal advice: SPDX ids such as `apache-2.0` and `mit` allow commercial use; `-nc-` means non-commercial; custom ids (`llama3.2`, `gemma`) point at a licence file with its own rules (Llama's caps very large companies and asks for attribution). Those go to a human.
- **Gate.** `manual` means you accept terms on the website and the owner (or an automatic check) approves your account; downloads then need your token.
- **Size.** Safetensors at 16-bit ≈ 2 bytes per parameter. Mistral's 27 GB is a trap: the repo ships the same weights twice (`consolidated.safetensors` and `model-*.safetensors`). Count only the files you will load.

---
## Part 2 · A shortlist with explicit rules, then a GGUF that fits

In [ ]:
# 🧪 Orbita Cloud's rules as code: commercial licence, no gate, 4-bit weights fit this machine.
def shortlist(models, ram_gb=RAM_GB, vram_gb=VRAM_GB):
    rows = []
    for m in models:
        gb16 = m.size_gb()
        gb4 = round(gb16 / 3.2, 1) if gb16 else None          # 4-bit GGUF ≈ 16-bit size / 3.2 (measured below)
        reasons = [r for r, bad in [("licence: " + m.verdict, m.verdict != "permissive"),
                                     ("gated", m.needs_token),
                                     ("too big", gb4 is not None and fits_in_memory(gb4, ram_gb, vram_gb=vram_gb) == "no")] if bad]
        rows.append({"model": m.id, "4-bit GB": gb4, "runs on": fits_in_memory(gb4, ram_gb, vram_gb=vram_gb) if gb4 else "?",
                     "decision": "✅ try it" if not reasons else "⛔ rejected (expected): " + ", ".join(reasons)})
    return rows


table(shortlist(pages.values()), ["model", "4-bit GB", "runs on", "decision"])
print()
table(shortlist(pages.values(), ram_gb=8), ["model", "4-bit GB", "runs on", "decision"])

**You should now see** two tables. On 32 GB: Qwen3-4B and Mistral 7B `✅ try it`, the rest `⛔ rejected (expected): …` for licence or gate (these are the shortlist doing its job, not errors). On an 8 GB laptop Mistral's estimate (from its doubled 27 GB) wrongly reads `too big`: the rule is only as good as the size you feed it.

### What just happened?

The decision is written down, so it can be reviewed and rerun when a new model appears. In a company you would add rules for language support, context window and the date of the last update, and store the table with the decision (the research log does that for the course).

In [ ]:
# 🌐 (LIVE) Real 4-bit sizes: the quantizations in a GGUF repo, and the Ollama name for each.
gguf = hub("Qwen/Qwen3-4B-GGUF")
table([{"quant": q, "GB": gb, "runs on": fits_in_memory(gb, RAM_GB, vram_gb=VRAM_GB), "ollama pull": ollama_hf_tag(gguf.id, q)}
       for q, gb in quant_files(gguf)], ["quant", "GB", "runs on", "ollama pull"])
print(f"\n16-bit {hub('Qwen/Qwen3-4B').size_gb()} GB → Q4_K_M {dict(quant_files(gguf))['Q4_K_M']} GB")

**You should now see** five files from `Q4_K_M` (2.33 GB) to `Q8_0` (3.99 GB), all `cpu`, each with a name like `hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M`, and `16-bit 7.49 GB → Q4_K_M 2.33 GB` (a factor of about 3.2).

### What just happened?

A GGUF repo holds one file per quantization. `Q4_K_M` is the usual default (4-bit, "K-quant", medium); `Q8_0` is almost lossless and nearly twice the size; below 4 bits quality drops fast on small models. Ollama can pull any public GGUF repo directly:

```powershell
ollama pull hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M
ollama run  hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M "Say hi"
```

That's also how you'll serve **your own fine-tuned model** in 14.04: upload the GGUF to the Hub (or import the file locally) and Ollama runs it. Use `MASTERY_MODEL_LOCAL=hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M` to point the course at it.

---
## Part 3 · Gated models and tokens

In [ ]:
# 🌐 (LIVE) Without a token, a gated file is refused; with HF_TOKEN (and approval) it downloads.
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError

if LIVE:
    try:
        path = hf_hub_download("meta-llama/Llama-3.2-1B-Instruct", "config.json")    # 1 KB file
        print("✅ downloaded (your token has access):", Path(path).name)
    except GatedRepoError as exc:
        print(f"⛔ GatedRepoError (expected without an approved token): {str(exc).splitlines()[0]}")
    except Exception as exc:
        print(f"⚠️ Hub unavailable ({type(exc).__name__}); in class this prints a 401 GatedRepoError.")
else:
    print("⏭️ Skipped: LIVE = False (offline). With network and no token this prints a 401 GatedRepoError.")
print("HF_TOKEN set:", HAVE_HF_TOKEN)

**You should now see** `⛔ GatedRepoError ... 401 Client Error` without a token (and `HF_TOKEN set: False`). With a token whose account Meta has approved, the file downloads.

### What just happened?

The **model page** of a gated repo is public (Part 1 read it); the **files** are not. A token identifies your account; it's useful only after you accept the terms on the model page. Create one at *huggingface.co → Settings → Access Tokens* with the **read** role (fine-grained if your organization requires it), put it in `.env` as `HF_TOKEN`, and never paste it into a notebook. `huggingface_hub`, `fastembed` and `transformers` all read `HF_TOKEN` automatically. Without a token you also get lower rate limits (the warning *"You are sending unauthenticated requests"*).

---
## Part 4 · A Hub model on your CPU with ONNX (no PyTorch)

In [ ]:
# 🌐 (LIVE) fastembed: a quantized ONNX copy of BAAI/bge-small-en-v1.5 from the Hub (67 MB, once), then offline.
import importlib.util

from mastery.memory import HashingEmbeddings

TICKETS = ["Our weather station stopped sending readings this morning.",
           "I was charged twice for the October invoice.",
           "How do I export my data as CSV?",
           "No data from the bridge since the storm, red light is on."]
QUERY = "station offline, no readings"


def cosine_rank(emb):
    q = emb.embed_query(QUERY)
    docs = emb.embed_documents(TICKETS)
    score = lambda d: sum(a * b for a, b in zip(q, d)) / ((sum(a * a for a in q) * sum(b * b for b in d)) ** 0.5 or 1)
    return sorted(((round(score(d), 2), t) for d, t in zip(docs, TICKETS)), reverse=True)


if LIVE and importlib.util.find_spec("fastembed"):
    from fastembed import TextEmbedding
    from mastery.memory import local_embeddings

    info = next(m for m in TextEmbedding.list_supported_models() if m["model"] == "BAAI/bge-small-en-v1.5")
    print("model:", info["model"], "| Hub repo:", info["sources"]["hf"], "| licence:", info["license"], "|", info["size_in_GB"], "GB")
    try:
        start = time.perf_counter()
        ranked = cosine_rank(local_embeddings())
        print(f"embedded 5 texts in {time.perf_counter() - start:.1f} s (first run includes the download)")
        for s, t in ranked:
            print(f"  {s:.2f}  {t}")
    except Exception as exc:
        print(f"⚠️ Could not download the model ({type(exc).__name__}); hashing baseline below.")
else:
    print("⏭️ Skipped: needs LIVE = True and fastembed (python -m pip install -e \".[embeddings]\").")
print("hashing baseline (words only):")
for s, t in cosine_rank(HashingEmbeddings()):
    print(f"  {s:.2f}  {t}")

**You should now see** the model line (`Qdrant/bge-small-en-v1.5-onnx-Q`, `mit`, 0.067 GB), then the two "no readings" tickets ranked on top by bge, while the hashing baseline only sees shared words: the bridge ticket gets 0.19 (one shared word) and the others 0.00.

### What just happened?

fastembed (Module 06) downloads from the Hub too: a **quantized ONNX export** of BAAI's model published by another account (Qdrant). That is common and worth checking: the licence is inherited (`mit`), but you trust the converter. ONNX Runtime runs it on the CPU without PyTorch, which is why this is the cheapest local win: small model, fast, private.

---
## Part 5 · Optional: `transformers` in-process on the CPU

In [ ]:
# 🖥️ (RUN_STRETCH) A 360M chat model with transformers + LangChain on the CPU. Needs the [transformers] extra.
RUN_STRETCH = False   # set True after: python -m pip install -e ".[transformers]"  (~200 MB+ of PyTorch)
SMALL = "HuggingFaceTB/SmolLM2-360M-Instruct"   # apache-2.0, 0.67 GB at 16-bit
if RUN_STRETCH and importlib.util.find_spec("transformers") and importlib.util.find_spec("langchain_huggingface"):
    from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

    start = time.perf_counter()
    pipe = HuggingFacePipeline.from_model_id(SMALL, task="text-generation",
                                             pipeline_kwargs={"max_new_tokens": 60, "do_sample": False, "return_full_text": False})
    print(f"loaded {SMALL} in {time.perf_counter() - start:.1f} s (first run downloads ~0.7 GB)")
    chat = ChatHuggingFace(llm=pipe)
    start = time.perf_counter()
    reply = chat.invoke("In one sentence: what is a support ticket queue?")
    print(f"{time.perf_counter() - start:.1f} s →", reply.content)
elif RUN_STRETCH:
    print('⏭️ Skipped: install the extra first: python -m pip install -e ".[transformers]", then restart the kernel.')
else:
    print("⏭️ Skipped (optional): set RUN_STRETCH = True to run a Hub model with transformers on your CPU.")

**You should now see** `⏭️ Skipped (optional)` by default. With the extra and `RUN_STRETCH = True`: a load time of about 10 s plus the first download, then one sentence in about a second.

### What just happened?

`HuggingFacePipeline` loads the 16-bit weights into **your Python process** with PyTorch; `ChatHuggingFace` applies the model's chat template so it behaves like any LangChain chat model. Compared with Ollama this is heavier (PyTorch, 16-bit, one process per notebook) but gives you the raw model: logits, tokenizer, and the exact weights you would fine-tune in 14.04. For **serving** on a CPU, a GGUF in Ollama is faster and lighter; for **research and training**, `transformers`.

In [ ]:
# 💸 (HAVE_HF_TOKEN, RUN_STRETCH) Hosted Inference Providers: same Hub model id, someone else's GPU. Paid.
if HAVE_HF_TOKEN and RUN_STRETCH:
    from huggingface_hub import InferenceClient

    client = InferenceClient(provider="auto")              # reads HF_TOKEN; picks an available provider
    start = time.perf_counter()
    out = client.chat.completions.create(model="openai/gpt-oss-20b", max_tokens=60,
                                         messages=[{"role": "user", "content": "In one sentence: what is a ticket queue?"}])
    print(f"{time.perf_counter() - start:.1f} s →", out.choices[0].message.content)
elif HAVE_HF_TOKEN:
    print("⏭️ Skipped (optional, paid): set RUN_STRETCH = True to try Inference Providers.")
else:
    print("⏭️ Skipped: HF_TOKEN is not set (optional; Inference Providers need credits).")

> 💸 Inference Providers bill per call at the provider's price. **Free Hugging Face accounts get no monthly credits** (verified 2026-10-16); PRO includes $2/month. The course never needs it: Groq's free tier serves the same `gpt-oss-20b`.

**You should now see** `⏭️ Skipped` (no token) or, with a token, credits and `RUN_STRETCH = True`, one sentence in a second or two.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `GatedRepoError: 401` | Gated repo, no token or not approved | Accept the terms on the model page, wait for approval, set `HF_TOKEN` in `.env` |
| `403` with a token | Fine-grained token without access to gated repos | Edit the token: enable *read access to public gated repos* |
| "You are sending unauthenticated requests" | No `HF_TOKEN` | Harmless warning; a read token raises your rate limits |
| Disk fills up | Hub cache in `%USERPROFILE%\.cache\huggingface` | `hf cache scan` / `hf cache delete`, or move it with `HF_HOME` |
| A repo looks twice as big as expected | Duplicate formats (`consolidated` + sharded, `.bin` + `.safetensors`) | Count only the files you load (`size_gb(".safetensors")`, one format) |
| `pip` resolves `huggingface-hub` back to 1.x | fastembed 0.9 and langchain-huggingface 1.2 require `<2` | Expected; the course pins `huggingface-hub>=1.33,<2` |
| `ollama pull hf.co/...` says not found | Repo has no GGUF, or the quant tag is wrong | Use a `-GGUF` repo and a tag from Part 2's table |

## ⚠️ Production notes

> ⚠️ **Production note: pin the revision, not just the name.** (Parts 1–2)
> A model repo can change under the same name. In production pass `revision="<commit sha>"` to `hf_hub_download` / `from_pretrained`, record it next to your evaluation results, and mirror the files you ship.

> ⚠️ **Production note: licences are a gate in your pipeline.** (Part 2)
> Put the shortlist rules in code review: a model with `read the terms` or `non-commercial` needs a named approver before it reaches users. Re-check when you switch to a fine-tune or a converted copy (Part 4): its licence and its trustworthiness come from whoever uploaded it.

## 🏋️ Exercises

1. **(Easy)** Search for `embedding` models (`task="feature-extraction"`). Which of the top five are permissive, and which is the smallest?
2. **(Medium)** Add a rule to `shortlist`: reject models whose last update is older than 18 months. (Hint: `HfApi().model_info(...).last_modified`.)
3. **(Medium)** Pull `hf.co/unsloth/Qwen3-1.7B-GGUF:Q4_K_M` with Ollama and run Lesson 14.01 Part 2 with it. Compare tokens per second with `qwen3:4b`.
4. **(Stretch)** Run Part 5 and time the same prompt on `ollama:qwen3:0.6b`. Which is faster on your CPU, and why?

<details>
<summary><b>Show solutions</b></summary>

**1.** `table([{"model": m.id, "verdict": m.verdict} for m in search_models("embedding", task="feature-extraction", limit=5)], ["model", "verdict"])`, then `model_details(id).size_gb()` for each. Expect Qwen3-Embedding, BGE and sentence-transformers models, mostly `apache-2.0`/`mit`.

**2.** `from datetime import datetime, timedelta, timezone` and, inside the loop, `info = HfApi().model_info(m.id)`, then add `("stale", info.last_modified < datetime.now(timezone.utc) - timedelta(days=548))` to `reasons`.

**3.** `ollama pull hf.co/unsloth/Qwen3-1.7B-GGUF:Q4_K_M`, then in 14.01 `llm = get_model(model="ollama:hf.co/unsloth/Qwen3-1.7B-GGUF:Q4_K_M", temperature=0)`. Expect about twice the tokens per second of the 4B model.

**4.** Ollama is usually faster: 4-bit GGUF weights mean a quarter of the memory traffic per token, and llama.cpp is tuned for CPUs; transformers runs 16-bit (or 32-bit) PyTorch kernels.
</details>

## 🔎 Docs drill (5 min)

On a gated model, what's the difference between `auto` and `manual` approval, and where does the user see the outcome?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Gated models](https://huggingface.co/docs/hub/models-gated).
2. **auto**: access is granted as soon as the user shares contact information and accepts the terms; **manual**: the repo owner reviews each request.
3. The user sees pending/accepted requests under *Settings → Gated Repositories*; only then does `HF_TOKEN` download the files.

</details>

## ✅ Checkpoint

- [ ] I can list Hub models and read licence, gate and size from Python, without a token.
- [ ] My shortlist rules are code, and I know why a size estimate can be wrong.
- [ ] I know the `hf.co/<repo>:<quant>` name that lets Ollama run any Hub GGUF.
- [ ] I can explain when an `HF_TOKEN` is needed and where it lives (`.env`, never the notebook).
- [ ] Restart & Run All passes offline (`LIVE = False`); outputs cleared before saving.

## 🔎 Research checkpoint

Find **one** model on the Hub for routing support tickets in Spanish and English that passes your shortlist, and log in [`docs/research-log.md`](../../docs/research-log.md): repo id, licence, gate, Q4_K_M size, and the Ollama name you would pull.

## Summary

- The Hub's public API tells you licence, gating, task and file sizes before you download anything.
- Write selection rules as code: commercial licence, no gate (or an approved one), fits your memory at 4-bit.
- GGUF repos hold one file per quantization; `Q4_K_M` ≈ 16-bit size / 3.2, and Ollama pulls them as `hf.co/<repo>:<quant>`.
- A token only helps after you're approved for a gated repo; keep it in `.env`.
- ONNX (fastembed) is the lightest local win; `transformers` gives the raw model; hosted providers cost money.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 14-03-when-to-fine-tune.ipynb** → [`14-03-when-to-fine-tune.ipynb`](14-03-when-to-fine-tune.ipynb).

[Module 14 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)